# Make training motif positions
Picks one motif hit per gene to hold the aSNV, and marks some train genes as rare-variant genes (effect allele frequency 0 during training).

Input: `data/{TF_NAME}.csv.gz`, all JASPAR 2024 hits of one TF's motif in each gene's TSS-centered window. Make it from the LAMP root with `get_tf_motif_csv.py` (about 5 minutes). For STAT1:
```bash
python scripts/motif_positions/get_tf_motif_csv.py --motif_id MA0137.4 --out data/STAT1.csv.gz
```
For another TF, pass its JASPAR motif ID (listed in `data/human_CORE_JASPAR_TFs.csv`), e.g. `--motif_id MA0035.5 --out data/GATA1.csv.gz`.

The released `data/motif_positions/STAT1_uniform_motif_positions_49kb.csv` is used for many of the runs in the paper. Running this notebook with the defaults gives a file like it (one randomly selected motif per gene), with the same genes. 

You can use this notebook as a starting point to generate your own custom motif_positions file, where the motifs perturbed by the aSNVs are sampled in a way that supports a particular experimental design (e.g., only motifs far from the gene)

In [ ]:
TF_NAME = 'STAT1'
WINDOW_SIZE = 49152  # bp, TSS-centered. Must match seq_len in the training config
MIN_SCORE = 400  # minimum JASPAR score of a motif hit. We used 400 in our manuscript, see the methods section for more details

MOTIF_CSV = f'../../data/{TF_NAME}.csv.gz'
N_RARE_VARIANT_GENES = 100
RARE_VARIANT_GENE_POOL = '../../data/gene_lists/train_genes_279.txt'  # rare-variant genes are sampled from these train genes (train_genes_279 was used for the released file)
OUT_CSV = f'../../data/motif_positions/{TF_NAME}_uniform_motif_positions_{WINDOW_SIZE // 1000}kb.csv'

In [ ]:
import os
import numpy as np
import pandas as pd

np.random.seed(0)
DATA_DIR = '../../data'
ENFORMER_SEQ_LEN = 196608

## Genes
Genes expressed in whole blood (`all_blood_genes.txt`), excluding genes that share a TSS bin (`gene1/gene2`) and genes with more than one TSS.

In [ ]:
regions = pd.read_csv(os.path.join(DATA_DIR, 'Enformer_genomic_regions_TSSCenteredGenes_FixedOverlapRemoval_NoDuplicatesNoOverlappingBins.csv'))
regions = regions[~regions['gene_name'].str.contains('/')]
regions = regions[~regions['gene_name'].isin(regions[regions['gene_name'].duplicated()]['gene_name'])]
blood_genes = set(open(os.path.join(DATA_DIR, 'gene_lists', 'all_blood_genes.txt')).read().split())
regions = regions[regions['gene_name'].isin(blood_genes)].rename(columns={'seqnames': 'chrom'})

# 1-based -> 0-based, and a WINDOW_SIZE window centered on the TSS
regions[['starts', 'ends', 'gene_start']] = regions[['starts', 'ends', 'gene_start']].astype(int) - 1
regions = regions.rename(columns={'starts': 'start_0', 'ends': 'end_0', 'gene_start': 'gene_start0'})
region_center = regions['end_0'] - ENFORMER_SEQ_LEN // 2
regions['short_start0'] = region_center - WINDOW_SIZE // 2
regions['short_end0'] = region_center + WINDOW_SIZE // 2
regions['set'].value_counts()

## Motif hits inside each window

In [ ]:
motif_df = pd.read_csv(MOTIF_CSV, index_col=0)
motif_df = motif_df.merge(regions[['gene_name', 'chrom', 'short_start0', 'short_end0', 'gene_start0', 'set']], on=['gene_name', 'chrom'])
motif_df = motif_df[motif_df['score'] >= MIN_SCORE]
motif_df = motif_df[(motif_df['chromStart'] >= motif_df['short_start0']) & (motif_df['chromEnd'] <= motif_df['short_end0'])]
print(f"{motif_df['gene_name'].nunique()} genes have at least one hit")

## Pick one motif per gene
Here one hit is picked at random per gene. This is the place to customise, e.g. keep only hits near or far from the TSS (`chromStart - gene_start0`), or pick the highest-scoring hit.

In [ ]:
positions = pd.concat([motif_df[motif_df['set'] == s].groupby('gene_name').sample(1) for s in ['train', 'valid', 'test']])
positions['set'].value_counts()

## Rare-variant genes
These train genes get an effect allele frequency of 0 during training, so their synthetic variant is only seen at validation/test.

In [ ]:
rare_pool = set(open(RARE_VARIANT_GENE_POOL).read().split())
rare_genes = positions[positions['gene_name'].isin(rare_pool)].sample(n=N_RARE_VARIANT_GENES, random_state=0)['gene_name']
positions['effect_allele_freq'] = 0.5


# if you do not want to include rare-variant genes, comment or delete the following line
positions.loc[positions['gene_name'].isin(rare_genes), 'effect_allele_freq'] = 0


positions = positions[['gene_name', 'chrom', 'chromStart', 'chromEnd', 'name', 'score', 'strand', 'TFName', 'short_start0', 'short_end0', 'gene_start0', 'set', 'effect_allele_freq']]
positions.to_csv(OUT_CSV, index=False)
positions.head()

# NOTE:
### If any changes made to the sampling of motifs and aSNVs in this notebook reduced the number of train,validation, and/or test genes, that will likely lead to errors during training because there is a mismatch in the expected usable genes and the genes that get an aSNV. You will need to create a line-delimited .txt file specifying which genes you wish to train/validate/test on

## Gene lists (optional)
Write train, validation, and test gene lists from the genes that got a motif position, split by Enformer's `set`. Pass them to `train.py` with `--train_gene_path`, `--valid_gene_path`, and `--test_gene_path`.

If you included rare variant genes (effect_allele_freq == 0) in the motif_positions file, those genes go in the train set. Valid and test genes must not overlap the train genes; `set` guarantees this.

In [ ]:
GENE_LIST_DIR = f'../../data/gene_lists/{TF_NAME}'
N_TRAIN_GENES = None  # e.g. 3000 to subsample train genes; None keeps all of them
os.makedirs(GENE_LIST_DIR, exist_ok=True)

def write_genes(genes, path):
    with open(path, 'w') as f:
        for gene in sorted(genes):
            f.write(gene + '\n')

train_genes = positions[positions['set'] == 'train']['gene_name'].unique()
if N_TRAIN_GENES is not None:
    rare = positions[positions['effect_allele_freq'] == 0]['gene_name'].unique()
    others = pd.Series(sorted(set(train_genes) - set(rare))).sample(n=N_TRAIN_GENES - len(rare), random_state=0)
    train_genes = list(rare) + others.tolist()
valid_genes = positions[positions['set'] == 'valid']['gene_name'].unique()
test_genes = positions[positions['set'] == 'test']['gene_name'].unique()

write_genes(train_genes, os.path.join(GENE_LIST_DIR, 'train_genes.txt'))
write_genes(valid_genes, os.path.join(GENE_LIST_DIR, 'valid_genes.txt'))
write_genes(test_genes, os.path.join(GENE_LIST_DIR, 'test_genes.txt'))
print(f"{len(train_genes)} train, {len(valid_genes)} valid, {len(test_genes)} test genes")